In [1]:
!pip install pandas sqlalchemy pymysql matplotlib seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 942.3 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.6 MB/s  0:00:00
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)

   ---------------------------------------- 0/4 [pymysql]
   ---------- ----------------------------- 1/4 [greenlet]
   ---------- ----------------------------- 1/4 [greenlet]
   -------------------- ------------------- 2/4 [sqlalchemy]
   -------------------- ------------------- 2/4 [sqlalchemy]
   -------------------- ------------------- 2/4 [sqlalchemy]
   -------------------- ------------------- 2/4 [sqlalchemy]
   ----------------

In [4]:
import pandas as pd
import numpy as np
import datetime
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
username = "root"
password = "selvamysql"
host = "localhost"
database = "sales_analysis"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")

print("Connected Successfully")


Connected Successfully


In [6]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_date,
    oi.quantity,
    oi.list_price
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
"""

data = pd.read_sql(query, engine)

data.head()


,order_id,customer_id,order_date,quantity,list_price
0,1,259,01-01-2016,1,599.99
1,1,259,01-01-2016,2,1799.99
2,1,259,01-01-2016,2,1549.00
3,1,259,01-01-2016,2,599.99
4,1,259,01-01-2016,1,2899.99


In [7]:
data["order_date"] = pd.to_datetime(data["order_date"])
data["total_value"] = data["quantity"] * data["list_price"]

data.head()


ValueError: time data "14-01-2016" doesn't match format "%m-%d-%Y", at position 9. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [8]:
print(data.columns)



Index(['order_id', 'customer_id', 'order_date', 'quantity', 'list_price'], dtype='object')


In [9]:
data.head()


,order_id,customer_id,order_date,quantity,list_price
0,1,259,01-01-2016,1,599.99
1,1,259,01-01-2016,2,1799.99
2,1,259,01-01-2016,2,1549.00
3,1,259,01-01-2016,2,599.99
4,1,259,01-01-2016,1,2899.99


In [10]:
# Convert order_date properly (DD-MM-YYYY format)
data["order_date"] = pd.to_datetime(data["order_date"], dayfirst=True)

# Create total_value column
data["total_value"] = data["quantity"] * data["list_price"]

data.head()


,order_id,customer_id,order_date,quantity,list_price,total_value
0,1,259,2016-01-01,1,599.99,599.99
1,1,259,2016-01-01,2,1799.99,3599.98
2,1,259,2016-01-01,2,1549.00,3098.00
3,1,259,2016-01-01,2,599.99,1199.98
4,1,259,2016-01-01,1,2899.99,2899.99


In [11]:
import datetime

# Create snapshot date (1 day after last order)
snapshot_date = data["order_date"].max() + datetime.timedelta(days=1)

rfm = data.groupby("customer_id").agg({
    "order_date": lambda x: (snapshot_date - x.max()).days,
    "order_id": "nunique",
    "total_value": "sum"
}).reset_index()

rfm.columns = ["customer_id", "recency", "frequency", "monetary"]

rfm.head()


,customer_id,recency,frequency,monetary
0,1,41,3,30645.87
1,2,264,3,21653.85
2,3,69,3,26249.81
3,4,255,3,24198.88
4,5,256,3,19442.88


In [12]:
# Recency score (lower recency = better customer)
rfm["R_score"] = pd.qcut(rfm["recency"], 4, labels=[4,3,2,1])

# Frequency score (higher frequency = better)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1,2,3,4])

# Monetary score (higher spending = better)
rfm["M_score"] = pd.qcut(rfm["monetary"], 4, labels=[1,2,3,4])

rfm.head()


,customer_id,recency,frequency,monetary,R_score,F_score,M_score
0,1,41,3,30645.87,4,4,4
1,2,264,3,21653.85,4,4,4
2,3,69,3,26249.81,4,4,4
3,4,255,3,24198.88,4,4,4
4,5,256,3,19442.88,4,4,4


In [13]:
rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

rfm["RFM_score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

rfm.head()


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,1,41,3,30645.87,4,4,4,444
1,2,264,3,21653.85,4,4,4,444
2,3,69,3,26249.81,4,4,4,444
3,4,255,3,24198.88,4,4,4,444
4,5,256,3,19442.88,4,4,4,444


In [14]:
def segment_customer(row):
    if row["R_score"] >= 3 and row["F_score"] >= 3 and row["M_score"] >= 3:
        return "High Value"
    elif row["R_score"] >= 3:
        return "Potential Loyalist"
    else:
        return "Low Value"

rfm["segment"] = rfm.apply(segment_customer, axis=1)

rfm.head()


,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score,segment
0,1,41,3,30645.87,4,4,4,444,High Value
1,2,264,3,21653.85,4,4,4,444,High Value
2,3,69,3,26249.81,4,4,4,444,High Value
3,4,255,3,24198.88,4,4,4,444,High Value
4,5,256,3,19442.88,4,4,4,444,High Value


In [15]:
# Keep only required columns for SQL table
final_rfm = rfm[[
    "customer_id",
    "recency",
    "frequency",
    "monetary",
    "segment"
]]

# Export to MySQL
final_rfm.to_sql


<bound method NDFrame.to_sql of       customer_id  recency  frequency  monetary     segment
0               1       41          3  30645.87  High Value
1               2      264          3  21653.85  High Value
2               3       69          3  26249.81  High Value
3               4      255          3  24198.88  High Value
4               5      256          3  19442.88  High Value
...           ...      ...        ...       ...         ...
1440         1441      337          1  10497.98  High Value
1441         1442      517          1   7841.94  High Value
1442         1443      774          1  11237.95   Low Value
1443         1444      739          1   1749.97   Low Value
1444         1445      297          1   9999.98  High Value

[1445 rows x 5 columns]>